# Collimator Mesh Definition

Written by Lev Thomson for python 3.9.25

This script generates a mesh for the thruster and collimator using gmsh's python API. 
The mesh can then be imported into LEoPart-x / FEniCSx for use in plasma simulation. 

# Packages / Libraries:

In [1]:
import gmsh # For creating mesh
import numpy as np # For adding magnets
import meshio # For converting to xdmf format for FEniCSx
from collections import defaultdict # For grouping unsorted entities
import json # For mapping names of entities to their tags for import into FEniCSx

# Variables

First, variables are defined for the thruster dimensions, the plasma's Debye length and the dimensions of the mesh. 

In [2]:
import variables
mode = "run" # "debug" for extra messages printed to screen

# Functions

The number of magnets used in the collimator varies, and is specified in n_mag. The function add_magnet_pair (defined below) is called repeatedly when generating the mesh. This adds an evenly-spaced set of n_mag*2 magnets around the inner rim of the collimator. 

In [3]:
def add_magnet_pair(
    model: gmsh.model,
    x: float,
    y: float,
    z: float,
    dx: float,
    dy: float,
    dz: float,
    angle: float,
    col_tag:int,
    magnet_volume_list_north: list,
    magnet_volume_list_south: list,
    N_total_pairs: int, 
    n_pair: int,
) -> tuple[gmsh.model, int, list, list]:
    """
    Adds a pair of magnets at a given point on the collimator-thruster mesh.

    The magnet is a rectangle defined by a point (x\', y\', z\') and the extents along the x-, y- and z-axes, and is rotated by angle.

    The second magnet is added diagonally opposite the first.

    Args:

        model: the model to be updated

        x: x-coordinate of one corner of box

        y: y-coordinate of one corner of box

        z: z-coordinate of one corner of box

        dx: x-coordinate of other corner of box

        dy: y-coordinate of other corner of box

        dz: z-coordinate of other corner of box

        angle: (radians) the rectangle is rotated about the x-axis by this angle

        col_tag: tag of the collimator that the magnets are attached to

        magnet_volume_list_north: list of all the tags of the north-polarised magnet volumes. For defining the magnets as a physical group later.

        magnet_volume_list_south: list of all the tags of the south-polarised magnet volumes. For defining the magnets as a physical group later.

        N_total_pairs: number of total magnet pairs in the collimator (determines polarity of magnets)

        n_pair: index of the pair being created when calling this function (determines polarity of magnets)

    Returns:

        model: updated gmsh model

        last_tag: the tag of the updated model

        magnet_surface_list: list of magnet surfaces by [(dimension=2, tag)] (for defining field later)
    """

    # Add first magnet

    mag_1_vol = model.occ.addBox(x, -y, -z, dx, dy, dz, -1) #+ 5e-4

    model.occ.rotate([(3, mag_1_vol)], x + dx / 2, 0, 0, 1, 0, 0, angle)

    # Add second magnet opposite the first

    mag_2_vol = model.occ.addBox(x, -y, -z, dx, dy, dz, -1) # + 5e-4

    model.occ.rotate([(3, mag_2_vol)], x + dx / 2, 0, 0, 1, 0, 0, angle + np.pi)

    # Fragment magnets with the collimator - this ensures that the mesh cells in the magnet and the collimator are conformal

    dt, dtmap = model.occ.fragment(
        [(3, col_tag)],
        [(3, mag_1_vol), (3, mag_2_vol)],
    )

    # The magnet and collimators are deleted and recreated when fragmenting. 
    # We then need to find their new tags programatically.

    # dtmap maps the new child volumes to their parents, and is organised like:
    # [
    #  [col_tag's children dimtags],
    #  [mag_1_vol's children dimtags],
    #  [mag_2_vol's children dimtags]
    #  ]

    col_tag = dtmap[0][0][1]
    # mag_1_vol = dtmap[1][0][1]
    # mag_2_vol = dtmap[2][0][1]

    gmsh.model.occ.synchronize()

    # magnet_volume_list.append(mag_1_vol)
    # magnet_volume_list.append(mag_2_vol)

    for mag in range(1, 3):
        # 1st magnet in pair is north if it's the 1st, 3rd, ... n_pair, south otherwise. 
        # 2nd magnet in pair matches 1st if N_total_pairs is even, otherwise it has the opposite polarity. 

        # Determine first magnet in pair polarity
        if n_pair % 2 == 1: # If odd pair
            polarity = -1
        else: # even pair
            polarity = 1

        if mag == 2: # if second magnet in pair
            if N_total_pairs % 2 == 1: # swap polarity if there are an odd number of total pairs
                polarity = -1 * polarity
            else:
                pass
        else:
            pass

        if polarity == 1:
            magnet_volume_list_north.extend(tag for _, tag in dtmap[mag])
        elif polarity == -1:
            magnet_volume_list_south.extend(tag for _, tag in dtmap[mag])

    model.occ.synchronize()

    return model, col_tag, magnet_volume_list_north, magnet_volume_list_south

def add_physical_boundary_surface(
    model: gmsh.model, vol_tag: int | list, phys_tag=int, surf_name=str
) -> int:
    """
    Adds a physical group containing the boundary of the volume vol_tag.

    The resulting physical group has the tag phys_tag, and the name surf_name.

    Args:

        - model: gmsh model to add the physical group to.

        - vol_tag: (positive integer) the tag of the volume whose surface will be added.

        - phys_tag: (positive integer) the tag of the resulting physical group. -1 to let gmsh assign a tag automatically.

        - surf_name: (string) the name of the physical group.



    Returns: the tag of the physical group, phys_tag.
    """
    if type(vol_tag) == int:
        boundary_entities = model.getBoundary([(3, vol_tag)])

        surface_tags = [tag for dim, tag in boundary_entities if dim == 2]
        return model.addPhysicalGroup(2, surface_tags, tag=phys_tag, name=surf_name)
    elif type(vol_tag) == list:
        surface_tags = []
        for vol in vol_tag:
            boundary_entities = model.getBoundary([(3, vol)])
            for dim, tag in boundary_entities:
                if dim == 2:
                    surface_tags.append(tag)
        return model.addPhysicalGroup(2, surface_tags, tag=phys_tag, name=surf_name)

def create_mesh(mesh, cell_type, prune_z=False):
    """
    Modified from code written by J.S. Dokken, December 2021, available here: (https://fenicsproject.discourse.group/t/importing-mesh-gives-error/7282/5)

    This reads in a mesh of type gmsh.mesh, and returns a mesh consisting of only the cells of type cell_type.

    Args:

        - mesh: gmsh.mesh containing mixed cell types

        - cell_type: either "tetra" or "triangle" - the cell type to be included in the output mesh.



    Returns:

        - mesh containing only cells of cell_type.
    """
    cells = mesh.get_cells_type(cell_type)

    cell_data = mesh.get_cell_data(
        "gmsh:physical", cell_type
    )  # For encoding facet tags (required to preserve physical groups)

    points = mesh.points[:, :2] if prune_z else mesh.points

    out_mesh = meshio.Mesh(
        points=points,
        cells={cell_type: cells},
        cell_data={"name_to_read": [cell_data.astype(np.int32)]},
    )

    return out_mesh

# Defining a Model Using GMSH

GMSH stores data as a model, which is an object that contains topological entities. 

These entities are:
Order 0 entities: Points
Order 1 entities: Curves, which are defined by multiple points
Order 2 entities: Surfaces, which are defined by multiple curves
Order 3 entities: Volumes, which are defined by multiple surfaces

Each of these entities is identified by a dimtag, which is a tuple with values (dimension, tag).
The dimension is the order of the entity, and the tag is a positive integer that is unique to that entity within that order. (e.g., [(3,1)] is not the same as [(2,1)]. The first is a volume but the second is a surface.) Tags can be defined when the entity is created, or can be generated automatically by GMSH. 

Various operations are available that allow a user to create a complex shape from simpler entities. These include cut and intersect to remove one entity from another, fuse to join entities, and fragment and getBoundary to divide an order-x entity into its constituent parts of order x-1.  

In [4]:
# ---- Set parameters ----


gmsh.initialize()

gmsh.model.add("collimator")  # Add model with name collimator


# ---- Define geometry ----

vol_coll = gmsh.model.occ.addCylinder(
    variables.l_a, 0, 0, variables.l_coll, 0, 0, variables.r_coll, -1
)  # Collimator cylinder, tag = 1

vol_a = gmsh.model.occ.addCylinder(
    0, 0, 0, variables.l_a, 0, 0, variables.r_a, -1
)  # Anode cylinder, tag = 2

# Then remove the cathode from the anode cylinder

vol_c = gmsh.model.occ.addCylinder(0, 0, 0, variables.l_c, 0, 0, variables.r_c, -1)

vol_trig = gmsh.model.occ.addCylinder(0, 0, 0, variables.l_trig, 0, 0, variables.r_trig, -1)

vol_fragment, vol_fragment_map = gmsh.model.occ.fragment(
    [(3, vol_coll)], 
    [
        (3, vol_a), 
        (3, vol_c),
        (3, vol_trig)
     ], 
     -1)

vol_coll = vol_fragment_map[0][0][1]
vol_a = vol_fragment_map[1][0][1]
vol_c = vol_fragment_map[2][0][1]
vol_trig = vol_fragment_map[3][0][1]
vol_c_trig = vol_fragment_map[3][1][1] 
# The trigger sits within the cathode 
# - this is the section of the trigger that overlaps with the cathode.

gmsh.model.occ.synchronize()


# Add magnets:

if variables.n_mag == 0:
    pass

elif variables.n_mag > 0:
    col_tag = vol_coll  # Tag of most-recently created entity

    angle_array = np.linspace(0, np.pi, variables.n_mag + 1)[
        0:variables.n_mag
    ]  # Generate array of equally spaced magnet pairs around spherical cathode surface

    magnet_volume_list_north = []
    magnet_volume_list_south = []

    for loop_idx, angle in enumerate(angle_array):
        gmsh.model, col_tag, magnet_volume_list_north, magnet_volume_list_south = add_magnet_pair(
            gmsh.model,
            variables.l_a,
            variables.r_coll,
            variables.w_mag / 2,
            variables.l_mag,
            variables.h_mag,
            variables.w_mag,
            angle,
            col_tag,
            magnet_volume_list_north,
            magnet_volume_list_south,
            variables.n_mag, 
            loop_idx
        )

    vol_coll = col_tag

gmsh.model.occ.synchronize()

magnet_names_north = []
magnet_names_south = []

if variables.n_mag > 0:
    for magtag in magnet_volume_list_north:
        magnet_names_north.append((3, magtag))

    for magtag in magnet_volume_list_south:
        magnet_names_south.append((3, magtag))

# Add cylinder around collimator
# Cylinder should be a few magnet-lengths long to permit A -> 0 at boundary.
# Note magnet length is the same as the collimator length.  

vol_area = gmsh.model.occ.addCylinder(
    0, 0, 0, variables.l_area, 0, 0, variables.r_area, -1
)

gmsh.model.occ.synchronize()

# Cut everything else from this mesh
vols_to_fragment = [
        (3, vol_coll), 
        (3, vol_a),
        (3, vol_c),
        (3, vol_trig),
        (3, vol_c_trig)
     ]

if variables.n_mag > 0:
    vols_to_fragment.extend(magnet_names_north)
    vols_to_fragment.extend(magnet_names_south)

vol_fragment, vol_fragment_map = gmsh.model.occ.fragment(
    [(3, vol_area)], 
    vols_to_fragment, 
     -1)

gmsh.model.occ.synchronize()

# Find tags from fragmented volume:
vol_area = vol_fragment_map[0][0][1]
vol_coll = vol_fragment_map[1][0][1]
vol_a = vol_fragment_map[2][0][1]
vol_c = vol_fragment_map[3][0][1]
vol_trig = vol_fragment_map[4][0][1]
vol_c_trig = vol_fragment_map[5][0][1]

magnet_surface_list_north = []
magnet_surface_list_south = []

if variables.n_mag > 0:
    # Assign tags back to the magnets
    n_north = len(magnet_names_north)
    magnet_names_north = [dt for sub in vol_fragment_map[6:6+n_north] for dt in sub]
    magnet_names_south = [dt for sub in vol_fragment_map[6+n_north:] for dt in sub]

    magnet_volume_list_north = [t for _, t in magnet_names_north]
    magnet_volume_list_south = [t for _, t in magnet_names_south]

    magnet_surface_list_north.append(gmsh.model.getBoundary(magnet_names_north, combined=False, oriented=False))
    flat_magnet_surface_tags_north = [tag for surf in magnet_surface_list_north for (dim, tag) in surf]

    magnet_surface_list_south.append(gmsh.model.getBoundary(magnet_names_south, combined=False, oriented=False))
    flat_magnet_surface_tags_south = [tag for surf in magnet_surface_list_south for (dim, tag) in surf]

# Defining Physical Groups

GMSH allows users to group entities into "physical groups". This is required to export the .msh file to an .xdmf format for use in FEniCS, but it also makes it possible to define properties for these physical groups. (e.g., all magnets can have the same magnetic properties, which can be different from the collimator wall.)

In [ ]:
# ---- Define physical groups -----

entities = gmsh.model.getEntities()

print("All Model Entities:")
for e in entities:
    dim = e[0]
    tag = e[1]
    if dim == 3:
        print(f"Entity dim: {dim}, tag: {tag}")

phys_col = gmsh.model.addPhysicalGroup(3, [vol_coll], -1, "collimator")

phys_a = gmsh.model.addPhysicalGroup(3, [vol_a], -1, "anode")

phys_c = gmsh.model.addPhysicalGroup(3, [vol_c], -1, "cathode")

phys_trig = gmsh.model.addPhysicalGroup(3, [vol_trig, vol_c_trig], -1, "trigger")

phys_mag_north = gmsh.model.addPhysicalGroup(3, magnet_volume_list_north, -1, "magnets_north")

phys_mag_south = gmsh.model.addPhysicalGroup(3, magnet_volume_list_south, -1, "magnets_south")

phys_area = gmsh.model.addPhysicalGroup(3, [vol_area], -1, "area_around_collimator")

phys_surf_col = add_physical_boundary_surface(
    gmsh.model, vol_coll, 54, "collimator_surface"
)

phys_surf_a = add_physical_boundary_surface(gmsh.model, vol_a, -1, "anode_surface")

phys_surf_c = add_physical_boundary_surface(gmsh.model, vol_c, -1, "cathode_surface")

phys_surf_trig = add_physical_boundary_surface(
    gmsh.model, [vol_trig, vol_c_trig], -1, "trigger_surface"
)

phys_surf_mag_north = gmsh.model.addPhysicalGroup(
    2, flat_magnet_surface_tags_north, -1, "magnets_surface_north"
)

phys_surf_mag_south = gmsh.model.addPhysicalGroup(
    2, flat_magnet_surface_tags_south, -1, "magnets_surface_south"
)

phys_surf_area = add_physical_boundary_surface(gmsh.model, vol_area, -1, "area_surface")

# Get the tag of the outer surface of the area, then assign that to a different group
max_area = 0
outer_tag = 0
area_outer = 2 * np.pi * (variables.r_area) * variables.l_area
boundary_entities = gmsh.model.getBoundary([(3, vol_area)])
# Find the surface with the biggest surface area of all the surface entities in vol_area
for dim, tag in boundary_entities:
    area = gmsh.model.occ.getMass(2, tag)
    if np.isclose(area, area_outer):
        outer_tag = tag
# Then assign that tag to a separate physical group
phys_outer = gmsh.model.addPhysicalGroup(2, [outer_tag], -1, "outer_surface")


All Model Entities:
Entity dim: 3, tag: 2
Entity dim: 3, tag: 3
Entity dim: 3, tag: 4
Entity dim: 3, tag: 5
Entity dim: 3, tag: 7
Entity dim: 3, tag: 8
Entity dim: 3, tag: 9
Entity dim: 3, tag: 10
Entity dim: 3, tag: 11
Entity dim: 3, tag: 12
Entity dim: 3, tag: 14
Entity dim: 3, tag: 15
Entity dim: 3, tag: 16
Entity dim: 3, tag: 17
Entity dim: 3, tag: 18
Entity dim: 3, tag: 19
Entity dim: 3, tag: 20
Entity dim: 3, tag: 21
Entity dim: 3, tag: 22
Entity dim: 3, tag: 23
Entity dim: 3, tag: 24
Entity dim: 3, tag: 25
Entity dim: 3, tag: 26
Entity dim: 3, tag: 27
area_outer = 0.10005972601683491
area of this piece is 0.00667588438887831
area of this piece is 1.239579423591385e-05
area of this piece is 1.2395794235913854e-05
area of this piece is 5.210288204317211e-08
area of this piece is 0.0002
area of this piece is 5.003128912355881e-06
area of this piece is 0.0002
area of this piece is 5.003128912358101e-06
area of this piece is 5.210288204308071e-08
area of this piece is 5.2102882043172

If there are ungrouped entities in the model, this can introduce errors when reading in the file in DOLFINx. Assigning all ungrouped entities to a physical group (sorted by dimension) resolves this. 

In [6]:
entities = gmsh.model.get_entities(-1)

# Find entities not already part of a physical group and add them to a physical group by dimension

ungrouped_entities = entities

misctags = defaultdict(list)

physicalgroups = gmsh.model.get_physical_groups(-1)

print("Physical groups")

for p in physicalgroups:
    print(f"{gmsh.model.get_physical_name(p[0], p[1])}, dim: {p[0]}, tag = {p[1]}")

    grouped_entities = gmsh.model.get_entities_for_physical_group(p[0], p[1])

    print(f"entities in {gmsh.model.get_physical_name(p[0], p[1])}: {grouped_entities}")

    for g in grouped_entities:
        dt = (p[0], int(g))

        if dt in ungrouped_entities:
            ungrouped_entities.remove(dt)


# Sort ungrouped entities by dimension

for d, e in ungrouped_entities:
    misctags[d].append(e)


phys_misc = defaultdict(int)

phys_names = ["points", "lines", "surfaces", "volumes"]

for ii in range(0, 4):
    if ii in misctags:
        phys_misc[ii] = gmsh.model.addPhysicalGroup(
            ii, misctags[ii], tag=-1, name=phys_names[ii]
        )

        print(
            f"Physical group created for ungrouped {phys_names[ii]}, contains these tags: {misctags[ii]}"
        )


    for t in misctags[ii]:
        dt = (ii, t)

        if dt in ungrouped_entities:
            ungrouped_entities.remove(dt)


# Check again for ungrouped entities: (expect this to print an empty list)

print(f"Ungrouped entities after grouping: {ungrouped_entities}")


gmsh.model.occ.synchronize()

Physical groups
collimator_surface, dim: 2, tag = 54
entities in collimator_surface: [  5  19  21  22  23  25  26  54  57  58  59  61  62  88  89  90  91  92
  93  94  95  96  97  98  99 100 101 102]
anode_surface, dim: 2, tag = 55
entities in anode_surface: [ 5  6  7  8  9 10 11]
cathode_surface, dim: 2, tag = 56
entities in cathode_surface: [ 8  9 12 13]
trigger_surface, dim: 2, tag = 57
entities in trigger_surface: [10 11 13 14 15 15]
magnets_surface_north, dim: 2, tag = 58
entities in magnets_surface_north: [ 21  25  26  31  32  33  34  35  36  37  38  39  40  41  42  54  61  62
  64  65  66  67  80  81  82  83  84  85  86  87  94  95  96 107 108 109
 110 111 112 113 114 115 116 117 118 113 114 115 116 117 118]
magnets_surface_south, dim: 2, tag = 59
entities in magnets_surface_south: [ 19  22  23  27  28  29  30  43  44  45  46  47  48  49  50  57  58  59
  68  69  70  71  72  73  74  75  76  77  78  79  92 100 101 103 104 105
 106 119 120 121 122 123 124 125 126 121 122 123 124 1

# Defining Fields

After the geometry of the thruster and collimator has been defined, then a field is applied to the model. This allows users to specify the mesh size programatically. For the plasma simulation, the mesh should be smallest where the magnetic field is strong. For this reason, a threshold field is applied to the mesh. This specifies a minimum mesh size, which occurs within distance DistMin to the magnet surfaces. The mesh size then increases up to a maximum size, which occurs at a distance DistMax from the magnet surfaces. 

Distmax is defined as the Debye length. This is the distance over which an ion's magnetic field has an effect, and is the largest size that a mesh cell should be in the plasma simulation. 

The mesh is then generated by GMSH. This can be a computationally intensive process, so it is done in parallel where possible. The resulting mesh is then saved in a .msh file for importing to FEniCSx. 

If the code is run on a computer that supports graphics, then it is displayed in the GMSH gui automatically. 

In [7]:
# ---- Define mesh field ----

# UP TO HERE 
# Set a larger field size on this big mesh

# Get magnet boundary

magnet_array_all = []

for mag_idx, magnet in enumerate(magnet_surface_list_north):
    for surf_idx, surface in enumerate(magnet):
        magnet_array_all.append(surface[1])

for mag_idx, magnet in enumerate(magnet_surface_list_south):
    for surf_idx, surface in enumerate(magnet):
        magnet_array_all.append(surface[1])


distance_field = gmsh.model.mesh.field.add("Distance", 1)

gmsh.model.mesh.field.setNumbers(1, "FacesList", magnet_array_all)

gmsh.model.mesh.field.setNumber(1, "Sampling", 1000)


threshold_field = gmsh.model.mesh.field.add("Threshold", 2)

gmsh.model.mesh.field.setNumber(
    2, "InField", distance_field
)  # Create a mesh field that defines mesh size based on proximity to magnets

gmsh.model.mesh.field.setNumber(2, "SizeMin", variables.size_min)

gmsh.model.mesh.field.setNumber(2, "SizeMax", variables.size_max)

gmsh.model.mesh.field.setNumber(2, "DistMin", 0)

gmsh.model.mesh.field.setNumber(2, "DistMax", 0.05)  # 50mm


#gmsh.model.mesh.field.setAsBackgroundMesh(2)

const_field = gmsh.model.mesh.field.add("MathEval")
gmsh.model.mesh.field.setString(const_field, "F", str(1.5*variables.size_max))  # Your large constant size

restrict_const = gmsh.model.mesh.field.add("Restrict")
gmsh.model.mesh.field.setNumber(restrict_const, "InField", const_field)
gmsh.model.mesh.field.setNumber(restrict_const, "RegionsList", vol_area)

# Combine the two restricted fields
final_mesh_field = gmsh.model.mesh.field.add("Min")
gmsh.model.mesh.field.setNumbers(final_mesh_field, "FieldsList", [threshold_field, restrict_const])

# Set as active background
gmsh.model.mesh.field.setAsBackgroundMesh(final_mesh_field)
 # UP TO HERE idk if this will work

gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)

gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)

gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)


gmsh.model.occ.synchronize()

# Map Entity Names to Tags

The entities (points, curves, surfaces and volumes) in the gmsh model are defined by their dimtags, which are tuples of unique positive integers. 
The tags are also saved as named variables here, to make the code more readable. 

The entity names are not exported in the MSH or XDMF files, so they are saved here for import into FEniCSx.

In [8]:
# Create a dictionary of entities and tags

physical_groups_dict = defaultdict()

physicalgroups = gmsh.model.get_physical_groups(-1)

for p in physicalgroups:
    name = gmsh.model.get_physical_name(p[0], p[1])
    dim = p[0]
    tag = p[1]
    grouped_entities = gmsh.model.get_entities_for_physical_group(dim, tag)
    entities = [int(e) for e in grouped_entities]

    physical_groups_dict[name] = {
        "dim": dim,
        "tag": tag,
        "entities": entities # list of entity tags
    }

# Export data to a JSON file to read in later
with open("physicalgroups.json", "w") as file:
    json.dump(physical_groups_dict, file, indent=4)



# Generate Mesh

In [9]:
# ---- Parallelize ----


gmsh.option.setNumber(
    "General.NumThreads", 5
)  # Use multithreading to generate mesh faster

gmsh.option.setNumber(
    "Mesh.Algorithm3D", 4
)  # Frontal-Delaunay (default. If this causes issues consider changing to 10 - HXT which is a version of Delaunay that is optimised for parallel meshing).


# ---- Generate Mesh ----

gmsh.option.setNumber("Geometry.Tolerance", 1e-8)

# Remove duplicates (these can cause mesh issues)
gmsh.model.mesh.remove_duplicate_nodes()
gmsh.model.mesh.remove_duplicate_elements()

gmsh.model.mesh.generate()  # (3) # Generate in 3d

gmsh.write("meshes/"+variables.meshname)  # Save mesh as meshname


# ---- Output to GUI ----


if variables.run_gui:  # Off by default for remote server.
    gmsh.fltk.run()


gmsh.finalize()


print("Done")

Info    : Removing duplicate mesh nodes...
Info    : Found 0 duplicate nodes 
Info    : No duplicate nodes found
Info    : Removing duplicate mesh elements...
Info    : Done removing duplicate mesh elements
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 5 (Line)
Info    : [  0%] Meshing curve 7 (Circle)
Info    : [  0%] Meshing curve 4 (Circle)
Info    : [  0%] Meshing curve 6 (Circle)
Info    : [  0%] Meshing curve 8 (Circle)
Info    : [  0%] Meshing curve 9 (Line)
Info    : [  0%] Meshing curve 10 (Circle)
Info    : [  0%] Meshing curve 11 (Line)
Info    : [  0%] Meshing curve 12 (Circle)
Info    : [  0%] Meshing curve 13 (Circle)
Info    : [  0%] Meshing curve 14 (Line)
Info    : [  0%] Meshing curve 17 (Line)
Info    : [  0%] Meshing curve 20 (Line)
Info    : [  0%] Meshing curve 22 (Line)
Info    : [  0%] Meshing curve 23 (Line)
Info    : [  0%] Meshing curve 24 (Line)
Info    : [  0%] Meshing curve 26 (Line)
Info    : [  0%] Meshing curve 27 (Line)
Info    : [  0%] Meshin

Info    : Optimization starts (volume = 0.000622257) with worst = 7.25633e-07 / average = 0.654521:
Info    : 0.00 < quality < 0.10 :       523 elements
Info    : 0.10 < quality < 0.20 :      2745 elements
Info    : 0.20 < quality < 0.30 :      9434 elements
Info    : 0.30 < quality < 0.40 :     15296 elements
Info    : 0.40 < quality < 0.50 :     11156 elements
Info    : 0.50 < quality < 0.60 :     22322 elements
Info    : 0.60 < quality < 0.70 :     66168 elements
Info    : 0.70 < quality < 0.80 :     84863 elements
Info    : 0.80 < quality < 0.90 :     28628 elements
Info    : 0.90 < quality < 1.00 :     11109 elements
Info    : 8579 edge swaps, 2358 node relocations (volume = 0.000622257): worst = 0.0182329 / average = 0.66981 (Wall 0.233961s, CPU 0.234175s)
Info    : 10329 edge swaps, 3061 node relocations (volume = 0.000622257): worst = 0.129743 / average = 0.672719 (Wall 0.320545s, CPU 0.320829s)
Info    : 10671 edge swaps, 3306 node relocations (volume = 0.000622257): worst = 0

Info    : Optimization starts (volume = 0.00579121) with worst = 7.45193e-08 / average = 0.62001:
Info    : 0.00 < quality < 0.10 :       928 elements
Info    : 0.10 < quality < 0.20 :      6077 elements
Info    : 0.20 < quality < 0.30 :     45752 elements
Info    : 0.30 < quality < 0.40 :     53964 elements
Info    : 0.40 < quality < 0.50 :     19432 elements
Info    : 0.50 < quality < 0.60 :     75908 elements
Info    : 0.60 < quality < 0.70 :    122323 elements
Info    : 0.70 < quality < 0.80 :    137882 elements
Info    : 0.80 < quality < 0.90 :     77109 elements
Info    : 0.90 < quality < 1.00 :     21115 elements
Info    : 33152 edge swaps, 9125 node relocations (volume = 0.00579121): worst = 0.00235442 / average = 0.645311 (Wall 0.945869s, CPU 0.944262s)
Info    : 39110 edge swaps, 10785 node relocations (volume = 0.00579121): worst = 0.0165591 / average = 0.649413 (Wall 1.19952s, CPU 1.19812s)
Info    : 39764 edge swaps, 11235 node relocations (volume = 0.00579121): worst = 0.

Info    : Writing 'meshes/collimator.msh'...
Info    : Done writing 'meshes/collimator.msh'
Done


# Convert to XDMF format

GMSH creates .msh files, which are optimised for use with GMSH only, and are not able to be parallelised. To use the mesh in FEniCSx, the file must be converted to an XDMF and H5 file format, with the volume and surface entities separated into tetra and facet files. This can be done using meshio. 

In [10]:
# This step reads in the msh file so that the cell can be run without the use of gmsh.

in_mesh = meshio.read("meshes/"+variables.meshname)

collimator_tetra = create_mesh(in_mesh, "tetra")
collimator_triangle = create_mesh(in_mesh, "triangle")


# Check that the nodes are sorted correctly.
# All nodes should be unique, and there should be no triangles that are not in the tetra grid.
# Unexpected results here indicate an issue with geometry (probably overlapping meshes). 

print("Number of nodes in grid:", collimator_tetra.points.shape)
tetra_nodes = np.unique(collimator_tetra.cells_dict["tetra"].flatten())
print(
    "Number of unique nodes in grid:",
    len(tetra_nodes),
)

triangle_indices = np.unique(collimator_triangle.cells_dict["triangle"].flatten())
print(
    "Number of triangle nodes that are not in tetra grid:",
    np.sum(np.invert(np.isin(triangle_indices, tetra_nodes))),
)

meshio.write("meshes/collimator_tetra.xdmf", collimator_tetra, compression=None)
meshio.write("meshes/collimator_tetra_facets.xdmf", collimator_triangle, compression=None)

# Mixed-cell file (not used in simulation - just for inspection)
!meshio convert meshes/collimator.msh meshes/collimator.xdmf

if mode == "debug":
    import meshio
    from scipy.spatial import cKDTree

    m = meshio.read("meshes/"+variables.meshname)
    pts = m.points
    tree = cKDTree(pts)
    dists, _ = tree.query(pts, k=2)
    print("duplicates already in raw .msh:", np.sum(dists[:, 1] < 1e-9))


Number of nodes in grid: (140722, 3)
Number of unique nodes in grid: 140722
Number of triangle nodes that are not in tetra grid: 0



The mesh is complete. From here, the collimator.msh file can be inspected using the GMSH gui (available at https://www.gmsh.info). 
The mesh is loaded into FEniCSx in the file boundary_conditions.ipynb. 